# Circulation Graph
Spatial connectivity through apertures (doors and windows).
Two rooms are connected only if a door or window exists between them.

## 1. Import Libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from topologicpy.Color import Color

## 2. Check TopologicPy Version

In [ ]:
print(Helper.Version())

## 3. Set Renderer

In [ ]:
renderer = "vscode"

## 4. Import OBJ Files
Load rooms and apertures (doors and windows) as separate OBJ files exported from Rhino.

In [ ]:
rooms = Topology.ByOBJPath(r"E:\softwares-4\graph-ml\assign-01\geometry\box-house.obj", selfMerge=True)
print("Rooms:", rooms)
print("Number of rooms:", len(rooms))

In [ ]:
# Load apertures — update filenames once exported from Rhino
doors = Topology.ByOBJPath(r"E:\softwares-4\graph-ml\assign-01\geometry\box-house-doors.obj", selfMerge=True)
windows = Topology.ByOBJPath(r"E:\softwares-4\graph-ml\assign-01\geometry\box-house-windows.obj", selfMerge=True)
apertures = doors + windows
print("Doors:", len(doors))
print("Windows:", len(windows))
print("Total apertures:", len(apertures))

## 5. Classify Spaces
Build cells from faces and assign color by room type from the Rhino name.

- **Red** — Classroom  
- **Orange** — Office  
- **Green** — Corridor  
- **Blue** — Stair  
- **Violet** — Bathroom  
- **Cyan** — Other / Lobby

In [ ]:
cells = []
selectors = []

for obj in rooms:
    d = Topology.Dictionary(obj)
    faces = Topology.Faces(obj)
    if len(faces) > 1:
        c = Cell.ByFaces(faces)
        c = Topology.RemoveCollinearEdges(c)
        s = Topology.InternalVertex(c)
        name = Dictionary.ValueAtKey(d, "name")
        if name and "Classroom" in name:
            color = "red"
        elif name and "Office" in name:
            color = "orange"
        elif name and "Corridor" in name:
            color = "green"
        elif name and "Stair" in name:
            color = "blue"
        elif name and "Bathroom" in name:
            color = "violet"
        else:
            color = "cyan"
        d = Dictionary.SetValuesAtKeys(d, ["color", "vertex_size"], [color, 15])
        s = Topology.SetDictionary(s, d)
        selectors.append(s)
        cells.append(c)
        print(name, "→", color)

print("\nNumber of cells:", len(cells))

## 6. Build CellComplex and Transfer Dictionaries

In [ ]:
cc = CellComplex.ByCells(cells)
cc = Topology.RemoveCoplanarFaces(cc)
cc = Topology.RemoveCollinearEdges(cc)

# Transfer the room dictionaries from selectors back onto the CellComplex cells
cc = Topology.TransferDictionariesBySelectors(cc, selectors, tranVertices=False,
                                               tranEdges=False, tranFaces=False,
                                               tranCells=True)
print("CellComplex:", cc)
print("Number of cells:", len(Topology.Cells(cc)))

## 7. Add Apertures to the CellComplex
Apertures (doors and windows) are matched to the shared faces of the CellComplex.
A face becomes an aperture-bearing face only if an aperture geometry intersects it.

In [ ]:
cc = Topology.AddApertures(cc, apertures, subTopologyType="face")
print("Apertures added to CellComplex.")

## 8. Show Geometry with Apertures

In [ ]:
Topology.Show(cc,
              vertexSizeKey="vertex_size",
              vertexColorKey="color",
              faceOpacity=0.2,
              backgroundColor="white",
              width=700,
              height=500,
              renderer=renderer)

## 9. Build the Circulation Graph
`useApertures=True` means two cells are only connected when a door or window sits on their shared face.
`toExteriorApertures=False` excludes connections to the outside.

In [ ]:
g_circ = Graph.ByTopology(cc,
                           direct=True,
                           useApertures=True,
                           toExteriorApertures=False)
print("Circulation graph:", g_circ)
print("Vertices:", len(Graph.Vertices(g_circ)))
print("Edges:", len(Graph.Edges(g_circ)))

## 10. Assign Visual Attributes

In [ ]:
# Vertices: inherit color from room type, fallback to red
for v in Graph.Vertices(g_circ):
    d = Topology.Dictionary(v)
    color = Dictionary.ValueAtKey(d, "color") or "red"
    d = Dictionary.SetValuesAtKeys(d, ["size", "color"], [18, color])
    Topology.SetDictionary(v, d)

# Edges: uniform dark style
for e in Graph.Edges(g_circ):
    d = Dictionary.ByKeysValues(["width", "color"], [3, "#333333"])
    Topology.SetDictionary(e, d)

## 11. Show Circulation Graph

In [ ]:
Topology.Show(g_circ,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              backgroundColor="white",
              width=700,
              height=500,
              renderer=renderer)

## 12. Show Circulation Graph Overlaid on Geometry

In [ ]:
Topology.Show(cc, g_circ,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.15,
              backgroundColor="white",
              width=700,
              height=500,
              renderer=renderer)